# 🚀 MotionAI Coach – Google Colab Pose Server

### ⚠️ BƯỚC QUAN TRỌNG ĐẦU TIÊN (ĐỔI SANG GPU):
1. Trên menu trên cùng, bấm: **Thời gian chạy (Runtime)** -> **Thay đổi loại thời gian chạy (Change runtime type)**.
2. Tại **Trình tăng tốc phần cứng (Hardware accelerator)**: Chọn **T4 GPU** (thay vì CPU).
3. Bấm **Lưu (Save)**.


### Bước 1: Kiểm tra phần cứng (GPU hoặc CPU)

In [ ]:
# Kiểm tra GPU. Nếu hiện bảng Tesla T4 là tuyệt vời! Nếu báo không tìm thấy nvidia-smi thì máy đang chạy CPU.
!nvidia-smi || echo '⚠️ Đang chạy trên CPU. Hãy vào Thời gian chạy -> Thay đổi loại thời gian chạy -> chọn T4 GPU để tăng tốc độ!'

### Bước 2: Cài đặt thư viện cần thiết

In [ ]:
!pip install -q fastapi uvicorn python-multipart pyngrok httpx ultralytics opencv-python-headless scipy numpy
print('✅ Đã cài đặt xong thư viện!')

### Bước 3: Tải mã nguồn dự án

In [ ]:
import os
if os.path.exists('/content/projectCNWed'):
    !cd /content/projectCNWed && git pull
else:
    !git clone https://github.com/trunghieu714/projectCNWed.git /content/projectCNWed
print('✅ Đã tải mã nguồn dự án thành công!')

### Bước 4: Thiết lập mô hình AI trích xuất Pose

In [ ]:
USE_GVHMR = False  # Đổi True nếu muốn clone thêm repo GVHMR

if USE_GVHMR:
    if not os.path.exists('/content/GVHMR'):
        !git clone https://github.com/doantrunghieu08/GVHMR-Demo.git /content/GVHMR
        %cd /content/GVHMR
        !pip install -r requirements.txt
    os.environ['POSE_COMMAND'] = 'python tools/demo/demo.py --video={video} --output_root={out_dir} -s'
    os.environ['POSE_SOURCE'] = 'gvhmr'
else:
    os.environ['POSE_COMMAND'] = ''
    os.environ['POSE_SOURCE'] = 'yolo'
    os.environ['YOLO_POSE_WEIGHTS'] = 'yolo11x-pose.pt'
    print('✅ Sử dụng mô hình YOLO Pose!')

### Bước 5: Khởi động Pose Server nội bộ trên cổng 8100

In [ ]:
import subprocess, time, httpx

!fuser -k 8100/tcp || true

cmd = 'PYTHONPATH=/content/projectCNWed/backend python3 -m ai_engine.remote_server.pose_server'
server_proc = subprocess.Popen(cmd, shell=True)
time.sleep(3)

try:
    res = httpx.get('http://127.0.0.1:8100/', timeout=5)
    print('✅ Pose Server đã chạy ONLINE trên cổng 8100:', res.json())
except Exception as e:
    print('❌ Lỗi:', e)

### Bước 6: Mở đường hầm kết nối (Tunnel URL ra Internet)
Chạy ô này để nhận đường link HTTPS kết nối trực tiếp về máy của bạn.

In [ ]:
# 1. Tải cloudflared nếu chưa có
!which cloudflared > /dev/null || (wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb > /dev/null)

# 2. Tắt tunnel cũ
!killall cloudflared 2>/dev/null || true

# 3. Khởi động cloudflared ghi log vào file
import os, re, time
logfile = '/content/cloudflared.log'
if os.path.exists(logfile):
    os.remove(logfile)

!nohup cloudflared tunnel --url http://127.0.0.1:8100 --logfile /content/cloudflared.log > /dev/null 2>&1 &

print('Đang lấy URL Cloudflare Tunnel, vui lòng đợi vài giây...')
tunnel_url = None
for _ in range(25):
    time.sleep(1)
    if os.path.exists(logfile):
        with open(logfile, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
            m = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
            if m:
                tunnel_url = m[-1]
                break

if tunnel_url:
    print('\n' + '=' * 75)
    print('🎉 THÀNH CÔNG! ĐÃ CÓ LINK KẾT NỐI SERVER AI TỪ XA:')
    print(f'👉 PUBLIC URL: {tunnel_url}')
    print('=' * 75)
    print('\n📋 HÃY COPY DÒNG DƯỚI ĐÂY DÁN VÀO FILE backend/.env TRÊN MÁY BẠN:')
    print(f'AI_REMOTE_URL={tunnel_url}')
    print('AI_ESTIMATOR_TYPE=remote,precomputed,yolo,mock')
    print('=' * 75)
else:
    print('⚠️ Chưa lấy được URL Cloudflare. Dưới đây là nội dung log:')
    if os.path.exists(logfile):
        with open(logfile, 'r', errors='ignore') as f: print(f.read())
    print('\n👉 Bạn có thể dùng Ngrok ở Bước 7 để thay thế!')

### (Tùy chọn) Bước 7: Dùng Ngrok nếu Cloudflare bị chặn mạng

In [ ]:
from pyngrok import ngrok
# Lấy token miễn phí tại: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = ''  # Điền token vào đây nếu muốn dùng ngrok

if NGROK_TOKEN.strip():
    ngrok.kill()
    ngrok.set_auth_token(NGROK_TOKEN.strip())
    public_url = ngrok.connect(8100)
    print('👉 Ngrok URL:', public_url.public_url)
    print(f'AI_REMOTE_URL={public_url.public_url}')
else:
    print('Bỏ qua ô này nếu Bước 6 đã in ra link Cloudflare thành công.')